In [2]:
from pathlib import Path
import subprocess
from collections import Counter

import pandas as pd

In [3]:
data = Path().resolve() / 'resfinder'

In [4]:
with open(data.parent / "input.txt", "r") as file:
    files = [Path(line.strip()) for line in file.readlines()]

In [5]:
# cmd = f"""
# source activate abricate \
#     && bsub \
#         -o {data / 'abricate.out'} \
#         -e {data / 'abricate.err'} \
#             "abricate \
#                 --db resfinder \
#                 --nopath \
#                 --minid 95 \
#                 --mincov 80 \
#                 -fofn {data.parent / 'input.txt'} \
#                 >> {data / 'resfinder.tab'}
#             "
# """
# subprocess.run(cmd, shell=True)

In [6]:
df = pd.read_table(data / 'resfinder.tab', sep='\t')

In [7]:
cfr_genes = df.groupby('SEQUENCE')['PRODUCT'].apply(lambda x: [y for y in x if 'cfr' in y.lower()]).reset_index(name='cfr_gene').explode('cfr_gene')

In [8]:
cfr_genes = cfr_genes.drop_duplicates()
cfr_genes = cfr_genes.dropna(subset="SEQUENCE")

In [9]:
missing_cfr = cfr_genes[cfr_genes['cfr_gene'].isna()]["SEQUENCE"].to_list()
target_files = [file for file in files if file.stem in missing_cfr or file.stem not in df["SEQUENCE"].to_list()]

In [10]:
# for file in target_files:
#     out = data / 'blast'
#     out.mkdir(exist_ok=True)

#     cmd = f"""
#     source activate blast \
#         && bsub \
#             -o {out / 'blast.out'} \
#             -e {out / 'blast.err'} \
#                 "tblastn \
#                     -task tblastn \
#                     -query {data.parent / 'ARO_3000202-protein.fasta'}\
#                     -subject {file}\
#                     -evalue 1e-5 \
#                     -qcov_hsp_perc 90 \
#                     -max_hsps 1 \
#                     -out {out / f'{file.stem}.csv'} \
#                     -outfmt '20 qseqid sseqid pident qcovhsp evalue'
#                 "
#     """
#     subprocess.run(cmd, shell=True)

In [11]:
def read_blast(file):
    df = pd.read_csv(file)
    
    if df.empty:
        df.loc[0] = None
    
    df["SEQUENCE"] = file.stem
    return df


In [12]:
tblastn = pd.concat([read_blast(file) for file in data.glob("blast/*.csv")], ignore_index=True)

In [13]:
tblastn = tblastn.sort_values("evalue")
tblastn = tblastn.groupby("SEQUENCE").head(1)

In [14]:
tblastn['cfr_gene'] = tblastn['qseqid'].str.split("|").str[-1] + "-like"

In [15]:
cfr_likes = dict(zip(tblastn["SEQUENCE"], tblastn["cfr_gene"]))

In [16]:
cfr_genes["cfr_gene"] = cfr_genes["cfr_gene"].fillna(cfr_genes.apply(lambda x: cfr_likes.get(x["SEQUENCE"], x["cfr_gene"]), axis=1))

In [17]:
merged = df.groupby('SEQUENCE')['PRODUCT'].apply(lambda x: ', '.join(sorted(set(x)))).reset_index(name='resfinder_profile')
merged = merged.drop_duplicates()

In [18]:
merged = merged.merge(cfr_genes, on='SEQUENCE', how='left')

In [19]:
merged

,SEQUENCE,resfinder_profile,cfr_gene
0,11035714_2,"cfr, fexA",cfr
1,11037564_2,"aadD, bleO, cfr",cfr
2,11039781_2,"cfr, fexA",cfr
3,11041990_3,"cfr, fexA",cfr
4,11042864_2,"cfr, erm(B), tet(L)",cfr
...,...,...,...
181,OQ434560.1,cfr,cfr
182,OR902856.1,"blaTEM-116, cfr",cfr
183,OR902859.1,"blaTEM-116, cfr",cfr
184,OX441725.1,"cfr, fexA",cfr


In [21]:
merged.to_csv(data /'resfinder_results.csv', index=False)